# nanochat d12 — strong reference baseline

This notebook runs a **pinned upstream nanochat d12** baseline suitable for RG optimizer comparisons.

nanochat uses depth 12 as its reference/tuning scale. At d12, upstream chooses a 768-wide transformer and transfers the tuned optimization recipe using its scaling-law / μP-style rules. We preserve the upstream architecture, initialization, separate embedding/unembedding/scalar/matrix learning rates, hybrid AdamW+Muon optimizer, automatic token horizon and batch size, LR/momentum/weight-decay schedules, BOS-aligned packing, and tokenizer pipeline.

Three independent seeds are run. The RG wrapper pins nanochat commit `92d63d4e8bb4df75c3b71618f31ddde2378b2bcd` and only changes the upstream hard-coded seed 42 to read `NANOCHAT_SEED`; it does not replace nanochat training logic.


In [ ]:
from pathlib import Path
import os, sys
import pandas as pd

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / 'baseline'
    if (candidate / 'rg_baselines').is_dir():
        ROOT = candidate
        break
    if (path / 'rg_baselines').is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError('Run from a clone of CalculatedContent/rg_optimizers.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rg_baselines.nanochat_reference import (
    DEFAULT_NANOCHAT_SEEDS, NANOCHAT_COMMIT, NanoChatD12Config,
    analyze_weightwatcher_checkpoints, collect_metrics, ensure_checkout,
    ensure_environment, prepare_data, run_seed,
)

WORK_ROOT = Path(os.environ.get('RG_NANOCHAT_WORK_ROOT', ROOT / 'nanochat_work')).expanduser().resolve()
CHECKOUT = WORK_ROOT / 'upstream_nanochat'
CACHE = Path(os.environ.get('NANOCHAT_BASE_DIR', WORK_ROOT / 'cache')).expanduser().resolve()
RUN_DIR = Path(os.environ.get('RG_BASELINE_RUN_ROOT', ROOT / 'runs')).expanduser().resolve() / 'nanochat_d12_reference'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
CONFIG = NanoChatD12Config()
SEEDS = DEFAULT_NANOCHAT_SEEDS
NPROC_PER_NODE = int(os.environ.get('RG_NANOCHAT_NPROC', '8'))
print('Pinned nanochat commit:', NANOCHAT_COMMIT)
print('d12 width:', CONFIG.model_dim, 'seeds:', SEEDS, 'GPU processes:', NPROC_PER_NODE)
display(pd.DataFrame([CONFIG.__dict__]))


## 1. Pin nanochat and create its environment

The checkout is detached at the audited commit. nanochat's own `uv` environment and dependency configuration are used.


In [ ]:
CHECKOUT = ensure_checkout(CHECKOUT)
ensure_environment(CHECKOUT, gpu=True)
print('nanochat checkout:', CHECKOUT)


## 2. Prepare upstream data and tokenizer

This follows nanochat's miniseries setup: 1000 dataset shards and a 32,768-token tokenizer trained from up to 2B characters. Run once; later replicates reuse the cache.


In [ ]:
prepare_data(CHECKOUT, CACHE, CONFIG)
print('nanochat cache:', CACHE)


## 3. Run three d12 reference replicates

These are full reference runs, not smoke tests. Upstream nanochat computes the training horizon from 12 tokens per scaling parameter, auto-computes total token batch size, and applies its internal depth/batch LR and weight-decay scaling. Checkpoints and validation are emitted every 250 steps; CORE is evaluated at the final step.


In [ ]:
logs = []
for seed in SEEDS:
    log_path = run_seed(CHECKOUT, CACHE, RUN_DIR, CONFIG, seed=seed, nproc_per_node=NPROC_PER_NODE)
    logs.append((seed, log_path))

metrics = collect_metrics(logs, RUN_DIR / 'training_metrics_all_seeds.csv')
display(metrics.tail(30))


## 4. Offline WeightWatcher analysis

WeightWatcher runs after training so spectral diagnostics do not contaminate timed baseline performance. Every saved checkpoint is analyzed with `ERG=True, randomize=True`; all returned columns are retained, including alpha, randomized correlation-trap fields, and ERG metrics when supplied by WeightWatcher.


In [ ]:
spectral_frames = []
for seed in SEEDS:
    frame = analyze_weightwatcher_checkpoints(
        CHECKOUT, CACHE, seed=seed, output_csv=RUN_DIR / f'weightwatcher_seed{seed}.csv'
    )
    spectral_frames.append(frame)
spectral = pd.concat(spectral_frames, ignore_index=True)
spectral.to_csv(RUN_DIR / 'weightwatcher_all_seeds.csv', index=False)
display(spectral.tail(50))


## Baseline contract

Do not silently update the nanochat commit in an optimizer comparison. A new upstream commit constitutes a new baseline version. The persisted logs, checkpoints, configuration snapshots, training/validation metrics, CORE score, and WeightWatcher diagnostics define the reference control.
